In [ ]:
import pandas as pd

# ۱. تعریف لیست نام فایل‌ها بر اساس عکسی که فرستادید
file_names = [
    'DEMO_J.xpt', 'BMX_J.xpt', 'BPX_J.xpt', 'DR1TOT_J.xpt', 
    'DR2TOT_J.xpt', 'DXX_J.xpt', 'HDL_J.xpt', 'PAQ_J.xpt', 
    'SLQ_J.xpt', 'TRIGLY_J.xpt'
]

# ۲. خواندن فایل دموگرافیک به عنوان فایل پایه
# (نکته: باید آدرس فولدر خود را جایگزین کنید، مثلاً در گوگل کولاب یا سیستم خودتان)
path = "./"  # اگر فایل‌ها کنار کد هستند
merged_df = pd.read_sas(path + file_names[0])

# ۳. ادغام زنجیره‌ای بقیه فایل‌ها بر اساس ستون SEQN
for file in file_names[1:]:
    print(f"در حال ادغام فایل: {file}...")
    next_df = pd.read_sas(path + file)
    
    # استفاده از how='left' برای حفظ تمام افراد فایل دموگرافیک
    merged_df = pd.merge(merged_df, next_df, on='SEQN', how='left')

# ۴. ذخیره فایل نهایی به صورت یک فایل اکسل یا CSV یکپارچه
merged_df.to_csv('NHANES_Merged_Data.csv', index=False)
print("ادغام با موفقیت انجام شد! فایل NHANES_Merged_Data.csv آماده است.")

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

print("در حال بارگذاری داده‌های اصلی...")
df = pd.read_csv('NHANES_Merged_Data.csv')

# ۱. فیلتر افراد بالای ۱۸ سال
df_clean = df[df['RIDAGEYR'] >= 18].copy()

# ۲. پر کردن داده‌های مفقود بیوشیمیایی و آزمایشگاهی با میانه (Imputation)
df_clean['DXDTOLE'] = df_clean['DXDTOLE'].fillna(df_clean['DXDTOLE'].median())
df_clean['LBDHDD'] = df_clean['LBDHDD'].fillna(df_clean['LBDHDD'].median())
df_clean['LBXTR'] = df_clean['LBXTR'].fillna(df_clean['LBXTR'].median())
df_clean['BMXWT'] = df_clean['BMXWT'].fillna(df_clean['BMXWT'].median())
df_clean['BMXHT'] = df_clean['BMXHT'].fillna(df_clean['BMXHT'].median())
df_clean['BMXBMI'] = df_clean['BMXBMI'].fillna(df_clean['BMXBMI'].median())

# ۳. ساخت شاخص ژنتیکی/متابولیکی (Genetic_Score) بین ۰ تا ۱۰۰
# ترکیب توده عضلانی بدون چربی (ژنتیکی) + فاکتورهای چربی خون
scaler = MinMaxScaler(feature_range=(0, 100))
df_clean['Raw_Genetic_Score'] = (df_clean['DXDTOLE'] / df_clean['DXDTOLE'].median()) + \
                                (df_clean['LBDHDD'] / df_clean['LBDHDD'].median()) - \
                                (df_clean['LBXTR'] / df_clean['LBXTR'].median())

df_clean['Genetic_Score'] = scaler.fit_transform(df_clean[['Raw_Genetic_Score']])
df_clean['Genetic_Score'] = df_clean['Genetic_Score'].round(2)

# ۴. مهندسی شاخص فعالیت بدنی (Activity_Score)
df_clean['Vigorous_Activity'] = df_clean['PAQ650'].apply(lambda x: 1 if x == 1 else 0)
df_clean['Moderate_Activity'] = df_clean['PAQ665'].apply(lambda x: 1 if x == 1 else 0)
df_clean['Activity_Score'] = df_clean['Vigorous_Activity'] * 2 + df_clean['Moderate_Activity']

# ۵. محاسبه پروتئین مورد نیاز به گرم (Protein_Requirement_g)
# بر اساس فرمول شخصی‌سازی شده فیزیولوژیک (وزن، فعالیت و امتیاز ژنتیکی)
df_clean['Protein_Requirement_g'] = df_clean['Weight_kg'] * (0.8 + (df_clean['Activity_Score'] * 0.2) + (df_clean['Genetic_Score'] / 500))
df_clean['Protein_Requirement_g'] = df_clean['Protein_Requirement_g'].round(2)

# ۶. مرتب‌سازی و انتخاب ستون‌های نهایی استاندارد
final_cols = {
    'SEQN': 'ID',
    'RIDAGEYR': 'Age',
    'RIAGENDR': 'Gender',
    'Weight_kg': 'Weight_kg',
    'Height_cm': 'Height_cm',
    'BMI': 'BMI',
    'Genetic_Score': 'Genetic_Score',
    'Activity_Score': 'Activity_Score',
    'Protein_Requirement_g': 'Protein_Requirement_g'
}

df_final = df_clean[list(final_cols.keys())].rename(columns=final_cols)
df_final['Gender'] = df_final['Gender'].map({1: 'Male', 2: 'Female'})

# ۷. ذخیره و خروجی گرفتن نهایی
output_file = 'NHANES_Final_Dataset_with_Genetic_Score.csv'
df_final.to_csv(output_file, index=False)

print(f"عملیات با موفقیت انجام شد! تعداد رکوردهای نهایی: {df_final.shape[0]}")
print(f"فایل شما به نام {output_file} در منوی سمت چپ کولاب آماده دانلود است.")

در حال بارگذاری داده‌های اصلی...


FileNotFoundError: [Errno 2] No such file or directory: 'NHANES_Merged_Data.csv'

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

print("در حال بارگذاری داده‌های اصلی...")
df = pd.read_csv('NHANES_Merged_Data.csv')

# ۱. فیلتر افراد بالای ۱۸ سال
df_clean = df[df['RIDAGEYR'] >= 18].copy()

# ۲. پر کردن داده‌های مفقود بیوشیمیایی و آزمایشگاهی با میانه (Imputation)
df_clean['DXDTOLE'] = df_clean['DXDTOLE'].fillna(df_clean['DXDTOLE'].median())
df_clean['LBDHDD'] = df_clean['LBDHDD'].fillna(df_clean['LBDHDD'].median())
df_clean['LBXTR'] = df_clean['LBXTR'].fillna(df_clean['LBXTR'].median())
df_clean['BMXWT'] = df_clean['BMXWT'].fillna(df_clean['BMXWT'].median())
df_clean['BMXHT'] = df_clean['BMXHT'].fillna(df_clean['BMXHT'].median())
df_clean['BMXBMI'] = df_clean['BMXBMI'].fillna(df_clean['BMXBMI'].median())

# ۳. ساخت شاخص ژنتیکی/متابولیکی (Genetic_Score) بین ۰ تا ۱۰۰
scaler = MinMaxScaler(feature_range=(0, 100))
df_clean['Raw_Genetic_Score'] = (df_clean['DXDTOLE'] / df_clean['DXDTOLE'].median()) + \
                                (df_clean['LBDHDD'] / df_clean['LBDHDD'].median()) - \
                                (df_clean['LBXTR'] / df_clean['LBXTR'].median())

df_clean['Genetic_Score'] = scaler.fit_transform(df_clean[['Raw_Genetic_Score']])
df_clean['Genetic_Score'] = df_clean['Genetic_Score'].round(2)

# ۴. مهندسی شاخص فعالیت بدنی (Activity_Score)
df_clean['Vigorous_Activity'] = df_clean['PAQ650'].apply(lambda x: 1 if x == 1 else 0)
df_clean['Moderate_Activity'] = df_clean['PAQ665'].apply(lambda x: 1 if x == 1 else 0)
df_clean['Activity_Score'] = df_clean['Vigorous_Activity'] * 2 + df_clean['Moderate_Activity']

# ۵. محاسبه پروتئین مورد نیاز به گرم (Protein_Requirement_g) با استفاده از ستون‌های اصلی NHANES
df_clean['Protein_Requirement_g'] = df_clean['BMXWT'] * (0.8 + (df_clean['Activity_Score'] * 0.2) + (df_clean['Genetic_Score'] / 500))
df_clean['Protein_Requirement_g'] = df_clean['Protein_Requirement_g'].round(2)

# ۶. مرتب‌سازی و تغییر نام ستون‌های نهایی به فرمت استاندارد و تمیز
final_cols = {
    'SEQN': 'ID',
    'RIDAGEYR': 'Age',
    'RIAGENDR': 'Gender',
    'BMXWT': 'Weight_kg',
    'BMXHT': 'Height_cm',
    'BMXBMI': 'BMI',
    'Genetic_Score': 'Genetic_Score',
    'Activity_Score': 'Activity_Score',
    'Protein_Requirement_g': 'Protein_Requirement_g'
}

df_final = df_clean[list(final_cols.keys())].rename(columns=final_cols)
df_final['Gender'] = df_final['Gender'].map({1: 'Male', 2: 'Female'})

# ۷. ذخیره و خروجی گرفتن نهایی
output_file = 'NHANES_Final_Dataset_with_Genetic_Score.csv'
df_final.to_csv(output_file, index=False)

print("-" * 50)
print(f"✅ عملیات با موفقیت انجام شد!")
print(f"📊 تعداد رکوردهای نهایی بزرگسالان: {df_final.shape[0]}")
print(f"📁 فایل شما به نام '{output_file}' تولید شد. لطفاً منوی سمت چپ کولاب را رفرش کنید تا آن را دانلود کنید.")
print("-" * 50)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

print("در حال ساخت دیتاست مستر با استانداردهای Senior ML Engineer...")

# بارگذاری فایل اصلی ادغام شده شما
df = pd.read_csv('NHANES_Merged_Data.csv')

# ۱. فیلتر افراد بزرگسال (بالای ۱۸ سال)
df_clean = df[df['RIDAGEYR'] >= 18].copy()

# ۲. پاک‌سازی داده‌ها و پر کردن مقادیر خالی با میانه (Imputation)
df_clean['DXDTOLE'] = df_clean['DXDTOLE'].fillna(df_clean['DXDTOLE'].median())
df_clean['DXDTOPF'] = df_clean['DXDTOPF'].fillna(df_clean['DXDTOPF'].median())
df_clean['LBDHDD'] = df_clean['LBDHDD'].fillna(df_clean['LBDHDD'].median())
df_clean['LBXTR'] = df_clean['LBXTR'].fillna(df_clean['LBXTR'].median())
df_clean['BMXWT'] = df_clean['BMXWT'].fillna(df_clean['BMXWT'].median())
df_clean['BMXHT'] = df_clean['BMXHT'].fillna(df_clean['BMXHT'].median())
df_clean['BMXBMI'] = df_clean['BMXBMI'].fillna(df_clean['BMXBMI'].median())

# ۳. مهندسی ویژگی‌های ساختاری بدن (Body Composition)
df_clean['Lean_Mass_kg'] = (df_clean['DXDTOLE'] / 1000).round(2) # تبدیل گرم به کیلوگرم
df_clean['Body_Fat_Percent'] = df_clean['DXDTOPF'].round(2)

# ۴. استخراج میزان مصرف پروتئین واقعی فعلی فرد برای مقایسه آینده در Agent
df_clean['Daily_Protein_Intake_g'] = df_clean[['DR1TPROT', 'DR2TPROT']].mean(axis=1)
df_clean['Daily_Protein_Intake_g'] = df_clean['Daily_Protein_Intake_g'].fillna(df_clean['Daily_Protein_Intake_g'].median()).round(2)

# ۵. محاسبه شاخص ژنتیکی (Genetic_Score) بین ۰ تا ۱۰۰
scaler = MinMaxScaler(feature_range=(0, 100))
df_clean['Raw_Genetic_Score'] = (df_clean['DXDTOLE'] / df_clean['DXDTOLE'].median()) + \
                                (df_clean['LBDHDD'] / df_clean['LBDHDD'].median()) - \
                                (df_clean['LBXTR'] / df_clean['LBXTR'].median())
df_clean['Genetic_Score'] = scaler.fit_transform(df_clean[['Raw_Genetic_Score']]).round(2)

# ۶. مهندسی سطح فعالیت به دو صورت عددی و متنی (نقشه ۴ سطحی استاندارد)
df_clean['Vigorous_Activity'] = df_clean['PAQ650'].apply(lambda x: 1 if x == 1 else 0)
df_clean['Moderate_Activity'] = df_clean['PAQ665'].apply(lambda x: 1 if x == 1 else 0)
df_clean['Activity_Score'] = df_clean['Vigorous_Activity'] * 2 + df_clean['Moderate_Activity'] + 1 # مقادیر بین ۱ تا ۴

def map_activity_level(score):
    if score == 1: return 'Sedentary'
    elif score == 2: return 'Light'
    elif score == 3: return 'Moderate'
    else: return 'Active'

df_clean['Activity_Level'] = df_clean['Activity_Score'].apply(map_activity_level)

# ۷. محاسبه برچسب نهایی هدف (Protein_Requirement_g) مبتنی بر بافت خالص عضلانی و وزن کل
df_clean['Protein_Requirement_g'] = (df_clean['Lean_Mass_kg'] * 1.2) + (df_clean['BMXWT'] * (df_clean['Activity_Score'] * 0.1)) + (df_clean['Genetic_Score'] / 10)
df_clean['Protein_Requirement_g'] = df_clean['Protein_Requirement_g'].round(2)

# ۸. نگاشت نهایی به جدول ایده‌آل و تغییر نام ستون‌ها
final_mapping = {
    'SEQN': 'ID',
    'RIDAGEYR': 'Age',
    'RIAGENDR': 'Gender',
    'BMXWT': 'Weight_kg',
    'BMXHT': 'Height_cm',
    'BMXBMI': 'BMI',
    'Body_Fat_Percent': 'Body_Fat_Percent',
    'Lean_Mass_kg': 'Lean_Mass_kg',
    'Activity_Level': 'Activity_Level',
    'Activity_Score': 'Activity_Score',
    'Daily_Protein_Intake_g': 'Daily_Protein_Intake_g',
    'Genetic_Score': 'Genetic_Score',
    'Protein_Requirement_g': 'Protein_Requirement_g'
}

df_final_master = df_clean[list(final_mapping.keys())].rename(columns=final_mapping)
df_final_master['Gender'] = df_final_master['Gender'].map({1: 'Male', 2: 'Female'})

# ذخیره به عنوان فایل مرجع ابدی پروژه
df_final_master.to_csv('NHANES_Master_Dataset.csv', index=False)

print("-" * 60)
print("✅ دیتاست جامع مرجع (NHANES_Master_Dataset.csv) با موفقیت ساخته شد!")
print(f"📊 ابعاد فایل نهایی: {df_final_master.shape[0]} سطر و {df_final_master.shape[1]} ستون")
print("-" * 60)
print(df_final_master.head(2).to_string())